## Create Parameter for file run date

In [0]:
from datetime import date, timedelta
from pyspark.sql.functions import *

yesterday = date.today() - timedelta(days=1)
yesterday_str = yesterday.strftime("%Y%m%d")

dbutils.widgets.text("file_run_date", yesterday_str)

file_run_date = dbutils.widgets.get("file_run_date")
print(file_run_date)

# from datetime import date, timedelta
# from pyspark.sql.functions import *
# from pyspark.sql.functions import col

# dbutils.widgets.text("file_run_date", "20260208")
# file_run_date = dbutils.widgets.get("file_run_date")

## Delete the data in case of re-run

In [0]:
%sql
delete from `spark-databricks-project-catalog`.silver.orders_unified where file_date = :file_run_date;
delete from `spark-databricks-project-catalog`.silver.customers_unified where file_date = :file_run_date;
delete from `spark-databricks-project-catalog`.silver.products_unified where file_date = :file_run_date;

### Orders Data

In [0]:
df_online_orders = spark.table("`spark-databricks-project-catalog`.bronze.online_orders")
df_online_orders_filtered = df_online_orders.filter(col("file_date") == file_run_date) #just to read the latest data
#i.e. incremental loading

#df_online_orders_filtered.show(4)

In [0]:
from pyspark.sql.functions import *

df_cleaned_online_orders = df_online_orders_filtered.select(
    trim(col("order_id")).alias("order_id"),
    to_timestamp("order_ts").alias("order_ts"),
    trim(col("customer_id")).alias("customer_id"),
    trim(col("sku_id")).alias("sku_id"),
    col("qty").cast("int").alias("qty"),
    col("list_price").cast("double").alias("list_price"),
    col("discount").cast("double").alias("discount"),
    trim(col("payment_mode")).alias("payment_mode"),
    lit('N/A').alias("store_num"),
    col("file_date"),
    trim(col("source")).alias("source_system"),
    col("ingest_ts").alias("ingestion_time")
)
#df_cleaned_online_orders.show(5)

In [0]:
df_offline_orders = spark.table("`spark-databricks-project-catalog`.bronze.offline_orders")
df_offline_orders_filtered = df_offline_orders.filter(col("file_date") == file_run_date)

#df_offline_orders_filtered.show(5)

In [0]:
from pyspark.sql.functions import *

df_cleaned_offline_orders = df_offline_orders_filtered.select(
    trim(col("txn_id")).alias("order_id"),
    to_timestamp("txn_time").alias("order_ts"),
    trim(col("customer_id")).alias("customer_id"),
    trim(col("sku")).alias("sku_id"),
    col("quantity").cast("int").alias("qty"),
    col("mrp").cast("double").alias("list_price"),
    col("discount").cast("double").alias("discount"),
    trim(col("payment_mode")).alias("payment_mode"),
    col("store_num"),
    col("file_date"),
    trim(col("source")).alias("source_system"),
    col("ingest_ts").alias("ingestion_time")
)  
#df_cleaned_offline_orders.show(5)    

In [0]:
df_orders = df_cleaned_online_orders.union(df_cleaned_offline_orders)
df_orders.write.mode("append") \
         .saveAsTable("`spark-databricks-project-catalog`.silver.orders_unified") 

# df = spark.table("`spark-databricks-project-catalog`.silver.orders_unified")   
# df.show(4)             

## Customers Data

In [0]:
from pyspark.sql.functions import lit, col, current_timestamp

df_offline_customers = spark.table("`spark-databricks-project-catalog`.bronze.offline_customers")
df_offline_customers_filtered = df_offline_customers.filter(col("file_date") == file_run_date)
#df_offline_customers_filtered.show(4)

phone_trimmed = trim(col("phone"))

df_offline_customers_filtered_clean = df_offline_customers_filtered \
                             .withColumn("phone", substring(phone_trimmed, length(phone_trimmed) - 9, 10)) \
                             .withColumn("name", trim(col("name")))                           

df_offline_customers_filtered_clean = df_offline_customers_filtered_clean.dropDuplicates(["phone"]) #- not required as the data is already deduped  
#df_offline_customers_filtered_clean.show(4)


In [0]:
df_online_customers = spark.table("`spark-databricks-project-catalog`.bronze.online_customers")
df_online_customers_filtered = df_online_customers.filter(col("file_date") == file_run_date)
#df_online_customers_filtered.show(4)
phone_trimmed = trim(col("mobile_number"))

df_online_customers_filtered_clean = df_online_customers_filtered \
                         .withColumn("mobile_number", substring(phone_trimmed, length(phone_trimmed) - 9, 10)) \
                         .withColumn("full name", trim(col("full_name")))      

df_online_customers_filtered_clean = df_online_customers_filtered_clean.dropDuplicates(["mobile_number"])
#df_online_customers_filtered_clean.show(4)



In [0]:
joined = df_online_customers_filtered_clean.alias("on") \
              .join(df_offline_customers_filtered_clean.alias("off"), \
               col("on.mobile_number") == col("off.phone") \
                   ,"full_outer")

#joined.show(5)                

In [0]:
from pyspark.sql.functions import col, coalesce

final_df = joined.select(
    coalesce(col("on.full_name"), col("off.name")).alias("full_name"),
    coalesce(col("on.email"), col("off.email")).alias("email"),
    coalesce(col("on.mobile_number"), col("off.phone")).alias("mobile_number"),
    coalesce(col("on.city"), col("off.city")).alias("city"),
    coalesce(col("on.state"), col("off.state")).alias("state"),
    coalesce(col("on.zipcode"), col("off.pincode")).alias("zipcode"),
    col("on.signup_date").cast('date').alias("signup_date"),
    #col("off.store_id").alias("store_id") not taken while unifying customer's data
    col("off.cust_id").alias("offline_customer_id"),
    col("on.customer_id").alias("online_customer_id"),
    coalesce(col("on.file_date"), col("off.file_date")).alias("file_date"),
    coalesce(col("on.source"), col("off.source")).alias("source"),
    coalesce(col("on.ingest_ts"), col("off.ingest_ts")).alias("ingest_ts")
)
#final_df.show(5)

In [0]:
final_df.write.mode("append") \
              .saveAsTable("`spark-databricks-project-catalog`.silver.customers_unified")

## Product's Data

In [0]:
from pyspark.sql.functions import col, coalesce

df_online_products = spark.table("`spark-databricks-project-catalog`.bronze.online_products")
df_online_products_filtered = df_online_products.filter(col("file_date") == file_run_date)

df_online_products_filtered =df_online_products_filtered.dropDuplicates(["sku_id"])

df_online_products_filtered = df_online_products_filtered.withColumn("product_name", trim(col("product_name")) )

In [0]:
df_offline_products = spark.table("`spark-databricks-project-catalog`.bronze.offline_products")
df_offline_products_filtered = df_offline_products.filter(col("file_date") == file_run_date)

df_offline_products_filtered = df_offline_products_filtered.dropDuplicates(["sku"])
df_offline_products_filtered = df_offline_products_filtered.withColumn("name", trim(col("name")))

In [0]:
from pyspark.sql.functions import col, coalesce

joined_prod = df_online_products_filtered.alias("on") \
              .join(df_offline_products_filtered.alias("off") \
              , col("on.sku_id") == col("off.sku") \
              , "full_outer"
            )
#joined_prod.show(5)       

In [0]:
from pyspark.sql.functions import col, coalesce

final_products_df = joined_prod.select(
    coalesce(col("on.sku_id"), col("off.sku")).alias("sku_id"),
    coalesce(col("on.product_name"), col("off.name")).alias("product_name"),
    coalesce(col("on.category"), col("off.category_name")).alias("category"),
    coalesce(col("on.brand"), col("off.brand_name")).alias("brand"),
    coalesce(col("on.price"), col("off.mrp_price")).cast('float').alias("price"),

    #only in online data
    col("on.currency").alias("currency"),
    col("on.is_active").alias("is_active"),

    #only in offline data
    col("off.tax_rate").cast('float').alias("tax_rate"),
    
    coalesce(col("on.file_date"), col("off.file_date")).alias("file_date"),
    coalesce(col("on.source"), col("off.source")).alias("source"),
    coalesce(col("on.ingest_ts"), col("off.ingest_ts")).alias("ingest_ts")   
)
#final_products_df.show(4)
final_products_df.write.mode("append").saveAsTable("`spark-databricks-project-catalog`.silver.products_unified")